# Examen Segundo Parcial — Análisis de Datos (CETI IDS)
**Registro:** 23310032
**Nombre:** Alfredo Alejandro Longoria Alcantar
**Grupo:** 7B
**Fecha:** 27/10/2026

## Caso: Heart Disease — prevalencia de `target = 1` por `sex × cp`
**Contexto.** Unidad de análisis de datos de una secretaría de salud. Se construye una tabla de contingencia
sexo × tipo de dolor torácico con la prevalencia de enfermedad cardiaca (`target = 1`) y se visualiza como heatmap
anotado con escala de color de baja a alta prevalencia.

**Diccionario usado:** `sex` (0 = mujer, 1 = hombre); `cp` tipo de dolor torácico (angina típica, angina atípica,
dolor no anginoso, asintomático); `target` (1 = enfermedad cardiaca presente).

In [ ]:
# === Celda 1. Carga y preparación del dataset Heart Disease ===
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

URLS = [
    "https://raw.githubusercontent.com/sassoftware/examples/main/data/heart_disease.csv",  # URL del examen
    "heart_disease.csv",   # respaldo: copia local si la URL no responde
    "heart.csv",
]
df = None
for fuente in URLS:
    try:
        df = pd.read_csv(fuente)
        print("Datos cargados desde:", fuente)
        break
    except Exception as e:
        print("No se pudo leer", fuente, "->", type(e).__name__)
if df is None:
    raise FileNotFoundError("Descarga el CSV y guárdalo junto al notebook como heart_disease.csv")

df.columns = df.columns.str.strip().str.lower()

# Algunas versiones del dataset traen 'num' (0-4) en lugar de 'target'
if "target" not in df.columns and "num" in df.columns:
    df["target"] = (df["num"] > 0).astype(int)

# Solo se necesitan sex, cp y target; se eliminan filas sin esas variables
df = df.dropna(subset=["sex", "cp", "target"]).copy()
df["target"] = (pd.to_numeric(df["target"], errors="coerce") > 0).astype(int)

# Etiquetas legibles (codificación estándar UCI/Kaggle)
mapa_sex = {0: "Mujer", 1: "Hombre"}
# Se muestra el código original junto a la etiqueta para que no haya ambigüedad.
if set(df["cp"].unique()) <= {1, 2, 3, 4}:          # versión UCI original (1-4)
    mapa_cp = {1: "1·Angina típica", 2: "2·Angina atípica", 3: "3·No anginoso", 4: "4·Asintomático"}
else:                                                # versión recodificada 0-3 (las frecuencias coinciden
    mapa_cp = {3: "3·Angina típica", 1: "1·Angina atípica",   # con UCI: 0 es el grupo más grande =
               2: "2·No anginoso", 0: "0·Asintomático"}       # asintomático; 3 el más pequeño = típica)

df["sexo"] = df["sex"].map(mapa_sex).fillna(df["sex"].astype(str))
df["dolor"] = pd.Categorical(df["cp"].map(mapa_cp).fillna(df["cp"].astype(str)),
                             categories=list(mapa_cp.values()), ordered=True)

print(df.shape)
print("Prevalencia global de target=1:", round(df["target"].mean() * 100, 1), "%")
df[["age", "sex", "cp", "chol", "thalach", "target"]].head()

In [ ]:
# === Celda 2. Tabla de contingencia sex × cp ===
# n por celda (tamaño de la base de cada prevalencia)
n_tab = pd.crosstab(df["sexo"], df["dolor"], margins=True, margins_name="Total")

# casos con target = 1 por celda
casos_tab = pd.crosstab(df["sexo"], df["dolor"], values=df["target"], aggfunc="sum",
                        margins=True, margins_name="Total").fillna(0).astype(int)

# prevalencia (%) = casos / n  -> equivalente a la media de target en cada celda
prev_tab = (pd.crosstab(df["sexo"], df["dolor"], values=df["target"], aggfunc="mean",
                        margins=True, margins_name="Total") * 100).round(1)

# Intervalo de confianza 95% de Wilson para medir la inestabilidad de cada prevalencia
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    den = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / den
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return (max(0, centro - margen) * 100, min(1, centro + margen) * 100)

filas = []
for s in n_tab.index.drop("Total"):
    for c in n_tab.columns.drop("Total"):
        n, k = int(n_tab.loc[s, c]), int(casos_tab.loc[s, c])
        li, ls = wilson(k, n)
        filas.append({"sexo": s, "cp": c, "n": n, "casos_target1": k,
                      "prevalencia_%": round(k / n * 100, 1) if n else np.nan,
                      "IC95_inf": round(li, 1), "IC95_sup": round(ls, 1),
                      "amplitud_IC": round(ls - li, 1)})
resumen = pd.DataFrame(filas)

UMBRAL_N = 30   # celdas con menos de 30 pacientes se consideran inestables
resumen["estable"] = np.where(resumen["n"] >= UMBRAL_N, "Sí", "NO (n<30)")

print("Pacientes por celda (n):");            display(n_tab)
print("Casos con target = 1:");               display(casos_tab)
print("Prevalencia de target = 1 (%):");      display(prev_tab)
print("Detalle con IC95% de Wilson:")
display(resumen.sort_values("n"))

### Justificación metodológica — tabla de contingencia

- **Por qué `crosstab` con `aggfunc="mean"` y no un conteo simple:** como `target` es binaria (0/1), su media dentro de
  cada celda *es* la prevalencia. Un conteo de casos sin dividir entre el total de la celda confundiría "más enfermos"
  con "más pacientes" (por ejemplo, hay muchos más hombres que mujeres en el dataset), así que la proporción es la
  medida comparable entre celdas.
- **Por qué se reportan también `n` y los casos:** una prevalencia sin su denominador no dice qué tan confiable es.
  50 % con 2/4 y 50 % con 60/120 se ven idénticos en el heatmap, pero el primero puede cambiar 25 puntos con un
  solo paciente más.
- **Por qué el intervalo de Wilson y no el de Wald (p ± 1.96·√(p(1-p)/n)):** Wald falla con *n* pequeño y con
  proporciones cercanas a 0 % o 100 % (puede dar límites negativos o mayores a 100 %, o amplitud cero si p = 1).
  Wilson se mantiene dentro de [0, 100] y es el recomendado justo para las celdas que queremos diagnosticar.
- **Por qué umbral n < 30:** es una regla práctica común para considerar estable una proporción; además, con n < 30
  el IC95 de Wilson supera típicamente ±15 puntos, demasiado amplio para tomar decisiones de gestión.
- **Por qué no se imputan ni se fusionan categorías en este paso:** el objetivo es *mostrar* la inestabilidad, no
  ocultarla. Fusionar categorías (p. ej. angina típica + atípica) es una opción válida, pero borra la diferencia
  clínica que se quiere evaluar; se deja como recomendación, no como transformación silenciosa.

In [ ]:
# === Celda 3. Heatmap anotado de prevalencia ===
prev = prev_tab.drop(index="Total", columns="Total")
n_cel = n_tab.drop(index="Total", columns="Total")
k_cel = casos_tab.drop(index="Total", columns="Total")

# Anotación: prevalencia + casos/n; se marca con * la celda inestable
anot = prev.copy().astype(object)
for s in prev.index:
    for c in prev.columns:
        n = n_cel.loc[s, c]
        marca = " *" if n < UMBRAL_N else ""
        anot.loc[s, c] = f"{prev.loc[s, c]:.1f}%{marca}\n({k_cel.loc[s, c]}/{n})"

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(prev, annot=anot, fmt="", cmap="YlOrRd", vmin=0, vmax=100,
            linewidths=1.5, linecolor="white",
            cbar_kws={"label": "Prevalencia de target = 1 (%)"}, ax=ax,
            annot_kws={"fontsize": 11})

# Recuadro punteado sobre celdas con pocos casos
for i, s in enumerate(prev.index):
    for j, c in enumerate(prev.columns):
        if n_cel.loc[s, c] < UMBRAL_N:
            ax.add_patch(plt.Rectangle((j + 0.04, i + 0.04), 0.92, 0.92, fill=False,
                                       edgecolor="black", lw=2, linestyle="--"))

ax.set_title("Prevalencia de enfermedad cardiaca (target = 1) por sexo y tipo de dolor torácico\n"
             f"* y borde punteado: celda con n < {UMBRAL_N} (prevalencia inestable)", fontsize=12)
ax.set_xlabel("Tipo de dolor torácico (cp)")
ax.set_ylabel("Sexo")
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("Celdas inestables (n <", UMBRAL_N, "):")
display(resumen[resumen["n"] < UMBRAL_N][["sexo", "cp", "n", "casos_target1",
                                         "prevalencia_%", "IC95_inf", "IC95_sup"]])

### Justificación metodológica — heatmap

- **Escala secuencial `YlOrRd` (amarillo → rojo):** la prevalencia es una magnitud de una sola dirección (de baja a
  alta), así que corresponde una paleta secuencial; una divergente (p. ej. `coolwarm`) sugeriría un punto medio
  "neutral" que no existe aquí. El rojo comunica riesgo de forma intuitiva para un público de salud.
- **`vmin=0, vmax=100` fijos:** si la escala se ajusta solo al rango observado, una diferencia pequeña parece enorme y
  los colores no son comparables con otros heatmaps del informe.
- **Anotación `prevalencia% (casos/n)`:** el color solo comunica el porcentaje; el texto agrega el denominador para
  que el lector vea de inmediato si el color descansa sobre 4 pacientes o sobre 100.
- **Asterisco y borde punteado en celdas con n < 30:** el heatmap da el mismo "peso visual" a todas las celdas, sin
  importar su tamaño. Marcar las celdas inestables evita que un rojo intenso basado en pocos casos se lea como un
  hallazgo sólido.

### Decisión a justificar: ¿qué celdas tienen muy pocos casos y cómo afecta la interpretación?

**Celdas con pocos casos.** La tabla `resumen` (ordenada por *n*) y el bloque final de la celda 3 las identifican
automáticamente. En este dataset las mujeres son minoría (≈ 1/3 de los pacientes) y los tipos de dolor *angina típica*
y *asintomático* se reparten de forma muy desigual, por lo que las celdas críticas son las de **mujeres** en las
categorías de dolor menos frecuentes: **Mujer × angina típica** (la categoría de dolor más rara, ≈ 4 pacientes) y
**Mujer × angina atípica** (≈ 18 pacientes), ambas por debajo de 30. Hombre × angina típica (≈ 19) también queda bajo
el umbral. En esas celdas el IC95 de Wilson abarca
decenas de puntos porcentuales: un paciente más o menos con `target = 1` mueve la prevalencia entre 5 y 25 puntos.
*(Los valores exactos de n, casos e IC son los que imprime la celda 2; cópialos aquí al ejecutar.)*

**Cómo se refleja en el heatmap.** Esas celdas pueden aparecer con colores extremos (rojo muy intenso o amarillo
muy claro, incluso 0 % o 100 %) no porque el riesgo real sea extremo, sino porque con pocos pacientes la proporción
salta en escalones grandes. Por eso:

1. Se leen como **señales a confirmar, no como hallazgos**; el asterisco y el borde punteado lo advierten.
2. La comparación sólida se hace entre celdas con n ≥ 30 (Hombre × asintomático, Hombre × no anginoso, Hombre × angina atípica,
   Mujer × asintomático y Mujer × no anginoso) y con las **marginales** (`Total`), que agregan más pacientes.
3. Si se requiere una cifra para decidir sobre una celda inestable, conviene reportarla con su IC, fusionarla con
   una categoría clínica afín o acumular más registros antes de actuar.

### Conclusión para la gestión en salud

El heatmap muestra que la prevalencia de enfermedad cardiaca **no depende solo del sexo ni solo del tipo de dolor,
sino de su combinación**: dentro de cada sexo la prevalencia cambia mucho según `cp`, y entre sexos el patrón no es
el mismo, por lo que un protocolo de triaje que use una sola de las variables subestimaría el riesgo en algunos grupos.
Las celdas con *n* grande y prevalencia alta son las que justifican acción inmediata: deben priorizarse en la
consulta de primer contacto con ECG y valoración cardiológica temprana, porque ahí un resultado positivo es la regla
y no la excepción.

En cambio, las celdas de mujeres con pocos pacientes no deben usarse para fijar metas ni recortar recursos: su color
en el heatmap es inestable y podría reflejar azar. La recomendación operativa es (1) no descartar riesgo cardiovascular
en mujeres por presentar un dolor "atípico" o no anginoso, ya que la evidencia de esa celda es insuficiente para
afirmar que es bajo; (2) aumentar el registro de casos femeninos en las unidades de salud para que la próxima
actualización del tablero tenga n ≥ 30 por celda; y (3) publicar siempre la prevalencia junto con su *n* e intervalo
de confianza, para que los directivos distingan un riesgo confirmado de una señal que aún requiere vigilancia.